# Compile Season Data

Get tournament matchup matrix for specified season

In [1]:
season = 2024

playin_losers = (  # remove play-in losers from seeding data
    1224,  # Howard
    1438,  # Virginia
    1286,  # Montana St
    1129,  # Boise St
)

model_path = '../data/models/mens/2025_03_15_model.pkl'
data_path = '../data/models/mens/2025_03_15_data.parquet'

season

2024

### Previous Tournament Results

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

df = pd.read_parquet(r'..\data\preprocessed\mens_kaggle\tournament_results.parquet')

df = df.loc[df['Season'] == season, :].reset_index(drop=True)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results
0,2024,1101,Abilene Chr,-1.0,-0.333333
1,2024,1102,Air Force,-1.0,-1.000000
2,2024,1103,Akron,-1.0,-0.666667
3,2024,1104,Alabama,2.0,1.333333
4,2024,1105,Alabama A&M,-1.0,-1.000000
...,...,...,...,...,...
375,2024,1476,Stonehill,-1.0,-1.000000
376,2024,1477,East Texas A&M,-1.0,-1.000000
377,2024,1478,Le Moyne,-1.0,-1.000000
378,2024,1479,Mercyhurst,-1.0,-1.000000


### Barttorvik Ratings

In [3]:
df_barttorvik = pd.read_parquet(r'..\data\preprocessed\mens_barttorvik\barttorvik.parquet')

df_barttorvik = df_barttorvik.loc[df_barttorvik['Season'] == season, :].reset_index(drop=True)

df_barttorvik

,Season,TEAM,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2024,Houston,118.5,85.3,33.2,0.978,0.882353,49.7,44.0,29.9,39.0,13.7,24.7,36.9,30.2,64.2,43.4,30.0,16.1,7.4,49.1,60.2,36.6,40.9,64.3,78.774,2.288,63.130,69.4,32.991
1,2024,Connecticut,126.8,94.1,32.7,0.969,0.911765,57.1,45.1,33.3,32.5,14.9,16.2,36.5,26.8,66.2,43.7,31.9,14.2,8.3,63.7,46.0,40.9,33.2,65.6,81.643,1.710,58.255,74.2,34.029
2,2024,Purdue,126.1,94.3,31.8,0.966,0.878788,56.0,47.7,42.8,23.0,16.5,14.0,37.4,24.7,69.0,48.1,31.4,9.6,6.0,64.5,55.1,35.0,37.2,68.6,83.272,1.854,52.251,72.1,35.470
3,2024,Auburn,120.5,92.1,28.4,0.957,0.794118,54.1,43.4,38.2,41.0,14.9,18.2,32.9,30.3,71.0,42.8,29.8,16.0,8.6,62.0,42.8,37.5,33.3,70.8,80.954,2.198,43.272,75.2,27.793
4,2024,Arizona,121.6,93.3,28.3,0.955,0.757576,55.0,48.7,36.7,25.7,16.1,18.1,36.3,23.1,73.4,47.8,33.4,9.0,8.6,59.1,52.6,32.6,38.2,73.1,81.443,1.979,76.327,71.9,30.384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357,2024,Stonehill,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832
358,2024,Saint Francis,93.3,117.3,-24.0,0.067,0.214286,47.2,53.0,32.6,35.4,21.2,17.1,32.9,31.3,66.4,52.9,35.4,10.1,11.8,52.6,52.0,35.1,37.1,66.4,79.692,0.327,0.200,60.6,8.567
359,2024,IU Indy,92.7,116.9,-24.2,0.065,0.103448,46.5,58.2,33.2,33.4,21.3,18.5,30.0,35.5,68.6,59.0,38.0,6.0,7.7,42.4,55.3,24.3,37.5,68.3,79.492,2.044,4.750,72.3,10.790
360,2024,Coppin St.,84.7,110.0,-25.3,0.047,0.068966,42.1,51.3,31.1,38.3,22.9,21.8,27.0,38.6,67.3,51.0,34.5,8.0,9.5,40.7,58.6,34.4,37.6,67.3,80.172,1.292,4.358,72.6,9.758


In [4]:
df_spellings = pd.read_csv(
    r'..\data\unprocessed\kaggle\MTeamSpellings.csv', 
    encoding='cp1252'  # fixes issue with fancy quotes
)

df_spellings.loc[df_spellings.shape[0]] = ['fdu', 1192]

df_spellings

,TeamNameSpelling,TeamID
0,a&m-corpus chris,1394
1,a&m-corpus christi,1394
2,abilene chr,1101
3,abilene christian,1101
4,abilene-christian,1101
...,...,...
1173,youngstown st.,1464
1174,youngstown state,1464
1175,youngstown-st,1464
1176,youngstown-state,1464


In [5]:
spelling_to_id = dict(zip(df_spellings['TeamNameSpelling'], df_spellings['TeamID']))

len(spelling_to_id)

1178

In [6]:
from fuzzywuzzy.fuzz import token_sort_ratio
from fuzzywuzzy import process
from tqdm.autonotebook import tqdm

def match_names(team_spellings, new_data_teams):
    df_match = pd.DataFrame(
        [
            [
                new_data_team,
                *process.extract(
                    new_data_team,
                    team_spellings,
                    scorer=token_sort_ratio,
                    limit=1
                )[0][:2]
            ] for new_data_team in tqdm(new_data_teams)
        ],
        columns=['New Data Team', 'Team Spelling', 'Match Score']
    ).sort_values('Match Score', ignore_index=True)

    team_to_spelling = dict(zip(df_match['New Data Team'], df_match['Team Spelling']))

    return df_match, team_to_spelling

C:\Users\mhugh\AppData\Local\Temp\ipykernel_12452\2578028529.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [7]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_barttorvik['TEAM'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Queens,Queens (NC),80
1,UT Rio Grande Valley,texas rio grande valley,88
2,Saint Francis,saint francis (ny),90
3,Texas A&M Commerce,tx a&m commerce,91
4,Cal St. Bakersfield,cal state bakersfield,92
5,Southeast Missouri St.,southeast missouri state,93
6,Mississippi Valley St.,mississippi valley state,93
7,Texas A&M Corpus Chris,texas a&m-corpus christi,96
8,UC Riverside,uc riverside,100
9,Green Bay,green bay,100


In [8]:
df_barttorvik.insert(1, 'TeamID', df_barttorvik['TEAM'].map(team_to_spelling).map(spelling_to_id))

df_barttorvik

,Season,TeamID,TEAM,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2024,1222,Houston,118.5,85.3,33.2,0.978,0.882353,49.7,44.0,29.9,39.0,13.7,24.7,36.9,30.2,64.2,43.4,30.0,16.1,7.4,49.1,60.2,36.6,40.9,64.3,78.774,2.288,63.130,69.4,32.991
1,2024,1163,Connecticut,126.8,94.1,32.7,0.969,0.911765,57.1,45.1,33.3,32.5,14.9,16.2,36.5,26.8,66.2,43.7,31.9,14.2,8.3,63.7,46.0,40.9,33.2,65.6,81.643,1.710,58.255,74.2,34.029
2,2024,1345,Purdue,126.1,94.3,31.8,0.966,0.878788,56.0,47.7,42.8,23.0,16.5,14.0,37.4,24.7,69.0,48.1,31.4,9.6,6.0,64.5,55.1,35.0,37.2,68.6,83.272,1.854,52.251,72.1,35.470
3,2024,1120,Auburn,120.5,92.1,28.4,0.957,0.794118,54.1,43.4,38.2,41.0,14.9,18.2,32.9,30.3,71.0,42.8,29.8,16.0,8.6,62.0,42.8,37.5,33.3,70.8,80.954,2.198,43.272,75.2,27.793
4,2024,1112,Arizona,121.6,93.3,28.3,0.955,0.757576,55.0,48.7,36.7,25.7,16.1,18.1,36.3,23.1,73.4,47.8,33.4,9.0,8.6,59.1,52.6,32.6,38.2,73.1,81.443,1.979,76.327,71.9,30.384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357,2024,1476,Stonehill,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832
358,2024,1383,Saint Francis,93.3,117.3,-24.0,0.067,0.214286,47.2,53.0,32.6,35.4,21.2,17.1,32.9,31.3,66.4,52.9,35.4,10.1,11.8,52.6,52.0,35.1,37.1,66.4,79.692,0.327,0.200,60.6,8.567
359,2024,1237,IU Indy,92.7,116.9,-24.2,0.065,0.103448,46.5,58.2,33.2,33.4,21.3,18.5,30.0,35.5,68.6,59.0,38.0,6.0,7.7,42.4,55.3,24.3,37.5,68.3,79.492,2.044,4.750,72.3,10.790
360,2024,1164,Coppin St.,84.7,110.0,-25.3,0.047,0.068966,42.1,51.3,31.1,38.3,22.9,21.8,27.0,38.6,67.3,51.0,34.5,8.0,9.5,40.7,58.6,34.4,37.6,67.3,80.172,1.292,4.358,72.6,9.758


In [9]:
df = pd.merge(
    df,
    df_barttorvik.drop(columns=['TEAM']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832
376,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953
377,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904
378,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Barttorvik Previous Seasons

In [10]:
df_barttorvik_prev = pd.read_parquet(r'..\data\preprocessed\mens_barttorvik_full_season\barttorvik_full_season.parquet')

df_barttorvik_prev = df_barttorvik_prev.loc[df_barttorvik_prev['Season'] == season, :].reset_index(drop=True)

df_barttorvik_prev

,Season,TEAM,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2024,Abilene Christian,0.442,-2.137,0.588750,3.158750
1,2024,Air Force,0.567,2.424,0.371000,-5.150750
2,2024,Akron,0.648,5.612,0.649000,5.652500
3,2024,Alabama,0.955,27.053,0.884250,19.621500
4,2024,Alabama A&M,0.212,-11.586,0.130000,-16.677000
...,...,...,...,...,...,...
362,2024,Wright St.,0.445,-2.010,0.569750,2.691000
363,2024,Wyoming,0.536,1.330,0.540750,1.777250
364,2024,Xavier,0.889,19.535,0.835500,14.911500
365,2024,Yale,0.753,9.970,0.684333,7.184667


In [11]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_barttorvik_prev['TEAM'].unique())

df_match.head(25)

  0%|          | 0/367 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Queens,Queens (NC),80
1,UT Rio Grande Valley,texas rio grande valley,88
2,Saint Francis,saint francis (ny),90
3,Texas A&M Commerce,tx a&m commerce,91
4,Winston Salem St.,winston-salem-state,91
5,Cal St. Bakersfield,cal state bakersfield,92
6,Southeast Missouri St.,southeast missouri state,93
7,Mississippi Valley St.,mississippi valley state,93
8,Texas A&M Corpus Chris,texas a&m-corpus christi,96
9,Rice,rice,100


In [12]:
df_barttorvik_prev.insert(1, 'TeamID', df_barttorvik_prev['TEAM'].map(team_to_spelling).map(spelling_to_id))

df_barttorvik_prev

,Season,TeamID,TEAM,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2024,1101,Abilene Christian,0.442,-2.137,0.588750,3.158750
1,2024,1102,Air Force,0.567,2.424,0.371000,-5.150750
2,2024,1103,Akron,0.648,5.612,0.649000,5.652500
3,2024,1104,Alabama,0.955,27.053,0.884250,19.621500
4,2024,1105,Alabama A&M,0.212,-11.586,0.130000,-16.677000
...,...,...,...,...,...,...,...
362,2024,1460,Wright St.,0.445,-2.010,0.569750,2.691000
363,2024,1461,Wyoming,0.536,1.330,0.540750,1.777250
364,2024,1462,Xavier,0.889,19.535,0.835500,14.911500
365,2024,1463,Yale,0.753,9.970,0.684333,7.184667


In [13]:
df = pd.merge(
    df,
    df_barttorvik_prev.drop(columns=['TEAM']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844,0.648,5.612,0.64900,5.65250
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310,0.955,27.053,0.88425,19.62150
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df.loc[df['Past Year BARTHAG'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
8,2024,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,1134,Brooklyn,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,1147,Centenary,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,1215,Hardin-Simmons,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188,2024,1289,Morris Brown,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,2024,1302,NE Illinois,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
226,2024,1327,Okla City,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### My Rankings

In [15]:
df_rankings = pd.read_parquet(fr'..\data\preprocessed\mens_my_rankings\my_rankings_{season}.parquet')

df_rankings.insert(0, 'Season', season)

df_rankings.drop(columns=['Strength'], inplace=True)

df_rankings

,Season,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2024,Purdue,2.803218,0.278570,1.231482,0.952911,68.432063
1,2024,Connecticut,2.713893,0.299654,1.239103,0.939449,66.202452
2,2024,Houston,2.676655,0.323859,1.172888,0.849029,64.698118
3,2024,Tennessee,2.511437,0.247362,1.153969,0.906607,69.835235
4,2024,Auburn,2.474543,0.278585,1.197508,0.918923,70.514388
...,...,...,...,...,...,...,...
357,2024,Virginia Military Institute,-2.046609,-0.234906,0.884818,1.119724,75.177482
358,2024,Stonehill,-2.112391,-0.217351,0.914092,1.131443,68.999473
359,2024,Coppin State,-2.225919,-0.255060,0.848227,1.103286,67.569549
360,2024,Mississippi Valley State,-2.635805,-0.326825,0.849480,1.176304,65.648910


In [16]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_rankings['Team'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Purdue,purdue,100
1,Loyola Marymount,loyola marymount,100
2,Binghamton,binghamton,100
3,Western Illinois,western illinois,100
4,Columbia,columbia,100
5,Portland State,portland state,100
6,Pennsylvania,pennsylvania,100
7,Florida Gulf Coast,florida gulf coast,100
8,Cal State Fullerton,cal state fullerton,100
9,Cal State Bakersfield,cal state bakersfield,100


In [17]:
df_rankings.insert(1, 'TeamID', df_rankings['Team'].map(team_to_spelling).map(spelling_to_id))

df_rankings

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2024,1345,Purdue,2.803218,0.278570,1.231482,0.952911,68.432063
1,2024,1163,Connecticut,2.713893,0.299654,1.239103,0.939449,66.202452
2,2024,1222,Houston,2.676655,0.323859,1.172888,0.849029,64.698118
3,2024,1397,Tennessee,2.511437,0.247362,1.153969,0.906607,69.835235
4,2024,1120,Auburn,2.474543,0.278585,1.197508,0.918923,70.514388
...,...,...,...,...,...,...,...,...
357,2024,1440,Virginia Military Institute,-2.046609,-0.234906,0.884818,1.119724,75.177482
358,2024,1476,Stonehill,-2.112391,-0.217351,0.914092,1.131443,68.999473
359,2024,1164,Coppin State,-2.225919,-0.255060,0.848227,1.103286,67.569549
360,2024,1290,Mississippi Valley State,-2.635805,-0.326825,0.849480,1.176304,65.648910


In [18]:
df_rankings.loc[df_rankings['TeamID'].isna(), :]

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo


In [19]:
df = pd.merge(
    df,
    df_rankings.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875,0.024283,-0.045249,1.003736,1.048985,69.446555
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075,-0.234301,-0.084381,1.049157,1.133538,63.080511
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844,0.648,5.612,0.64900,5.65250,0.457533,0.053016,1.061068,1.008052,67.044145
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310,0.955,27.053,0.88425,19.62150,2.258407,0.228423,1.234926,1.006503,73.746625
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700,-0.879836,-0.137497,0.935035,1.072533,71.310687
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.112391,-0.217351,0.914092,1.131443,68.999473
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.182077,-0.167152,0.935817,1.102969,67.356915
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.158761,-0.089166,1.000531,1.089697,68.308798
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df.loc[df['Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
8,2024,1109,Alliant Intl,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,1118,Armstrong St,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,1121,Augusta,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,1128,Birmingham So,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,1134,Brooklyn,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,1147,Centenary,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,1215,Hardin-Simmons,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,2024,1216,Hartford,-1.0,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.042,-28.007,0.26400,-11.57475,NaN,NaN,NaN,NaN,NaN
188,2024,1289,Morris Brown,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,2024,1302,NE Illinois,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Starters

In [21]:
df_starters = pd.read_parquet(fr'..\data\preprocessed\mens_starters\starters_{season}.parquet')

df_starters.insert(0, 'Season', season)

df_starters.rename(columns={'Rating': 'Starters'}, inplace=True)

df_starters

,Season,Team,Starters
0,2024,Connecticut,0.548104
1,2024,Purdue,0.536821
2,2024,Houston,0.496831
3,2024,Auburn,0.483800
4,2024,North Carolina,0.441960
...,...,...,...
357,2024,Stonehill,-0.400817
358,2024,Virginia Military Institute,-0.406939
359,2024,Buffalo,-0.422366
360,2024,Mississippi Valley State,-0.442076


In [22]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_starters['Team'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Texas A&M-Commerce,tx a&m commerce,91
1,Connecticut,connecticut,100
2,Miami (OH),miami (oh),100
3,East Carolina,east carolina,100
4,Saint Louis,saint louis,100
5,Cal State Bakersfield,cal state bakersfield,100
6,Columbia,columbia,100
7,Binghamton,binghamton,100
8,Gardner-Webb,gardner webb,100
9,Canisius,canisius,100


In [23]:
df_starters.insert(1, 'TeamID', df_starters['Team'].map(team_to_spelling).map(spelling_to_id))

df_starters

,Season,TeamID,Team,Starters
0,2024,1163,Connecticut,0.548104
1,2024,1345,Purdue,0.536821
2,2024,1222,Houston,0.496831
3,2024,1120,Auburn,0.483800
4,2024,1314,North Carolina,0.441960
...,...,...,...,...
357,2024,1476,Stonehill,-0.400817
358,2024,1440,Virginia Military Institute,-0.406939
359,2024,1138,Buffalo,-0.422366
360,2024,1290,Mississippi Valley State,-0.442076


In [24]:
df_starters.loc[df_starters['TeamID'].isna(), :]

,Season,TeamID,Team,Starters


In [25]:
df = pd.merge(
    df,
    df_starters.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875,0.024283,-0.045249,1.003736,1.048985,69.446555,0.063427
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075,-0.234301,-0.084381,1.049157,1.133538,63.080511,-0.191559
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844,0.648,5.612,0.64900,5.65250,0.457533,0.053016,1.061068,1.008052,67.044145,0.234113
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310,0.955,27.053,0.88425,19.62150,2.258407,0.228423,1.234926,1.006503,73.746625,0.283066
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700,-0.879836,-0.137497,0.935035,1.072533,71.310687,-0.074870
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.112391,-0.217351,0.914092,1.131443,68.999473,-0.400817
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.182077,-0.167152,0.935817,1.102969,67.356915,-0.111816
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.158761,-0.089166,1.000531,1.089697,68.308798,-0.070744
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df.loc[df['Starters'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
8,2024,1109,Alliant Intl,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,1118,Armstrong St,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,1121,Augusta,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,1128,Birmingham So,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,1134,Brooklyn,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,1147,Centenary,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,1215,Hardin-Simmons,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,2024,1216,Hartford,-1.0,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.042,-28.007,0.26400,-11.57475,NaN,NaN,NaN,NaN,NaN,NaN
188,2024,1289,Morris Brown,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,2024,1302,NE Illinois,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Openskill Ratings

In [27]:
df_os = pd.read_parquet(fr'..\data\preprocessed\mens_os_rankings\os_rankings_{season}.parquet')

df_os.insert(0, 'Season', season)

df_os.drop(columns=['Sigma'], inplace=True)

df_os

,Season,Team,Mu,OS Rating
0,2024,Connecticut,51.086743,38.595099
1,2024,Houston,50.517837,38.093567
2,2024,Iowa State,48.187216,36.334127
3,2024,Purdue,49.322780,36.220702
4,2024,Auburn,47.461160,35.736291
...,...,...,...,...
357,2024,IU Indy,1.774340,-12.158040
358,2024,Virginia Military Institute,2.390031,-12.570655
359,2024,Detroit Mercy,1.112006,-13.481287
360,2024,Coppin State,0.378514,-13.655555


In [28]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_os['Team'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Connecticut,connecticut,100
1,Boston University,boston university,100
2,Tennessee State,tennessee state,100
3,North Florida,north florida,100
4,Cal State Fullerton,cal state fullerton,100
5,Texas Southern,texas southern,100
6,Mount St. Mary's,mount st. mary's,100
7,Middle Tennessee,middle tennessee,100
8,Canisius,canisius,100
9,Radford,radford,100


In [29]:
df_os.insert(1, 'TeamID', df_os['Team'].map(team_to_spelling).map(spelling_to_id))

df_os

,Season,TeamID,Team,Mu,OS Rating
0,2024,1163,Connecticut,51.086743,38.595099
1,2024,1222,Houston,50.517837,38.093567
2,2024,1235,Iowa State,48.187216,36.334127
3,2024,1345,Purdue,49.322780,36.220702
4,2024,1120,Auburn,47.461160,35.736291
...,...,...,...,...,...
357,2024,1237,IU Indy,1.774340,-12.158040
358,2024,1440,Virginia Military Institute,2.390031,-12.570655
359,2024,1178,Detroit Mercy,1.112006,-13.481287
360,2024,1164,Coppin State,0.378514,-13.655555


In [30]:
df = pd.merge(
    df,
    df_os.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875,0.024283,-0.045249,1.003736,1.048985,69.446555,0.063427,23.846923,11.768719
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075,-0.234301,-0.084381,1.049157,1.133538,63.080511,-0.191559,16.689482,3.046389
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844,0.648,5.612,0.64900,5.65250,0.457533,0.053016,1.061068,1.008052,67.044145,0.234113,31.015618,17.953874
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310,0.955,27.053,0.88425,19.62150,2.258407,0.228423,1.234926,1.006503,73.746625,0.283066,40.127890,28.383921
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700,-0.879836,-0.137497,0.935035,1.072533,71.310687,-0.074870,15.888827,3.620456
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.112391,-0.217351,0.914092,1.131443,68.999473,-0.400817,2.053793,-11.474784
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.182077,-0.167152,0.935817,1.102969,67.356915,-0.111816,12.329532,-0.371308
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.158761,-0.089166,1.000531,1.089697,68.308798,-0.070744,16.213913,3.887557
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
df.loc[df['OS Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating
8,2024,1109,Alliant Intl,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,1118,Armstrong St,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,1121,Augusta,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,1128,Birmingham So,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,1134,Brooklyn,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,1147,Centenary,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,1215,Hardin-Simmons,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,2024,1216,Hartford,-1.0,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.042,-28.007,0.26400,-11.57475,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188,2024,1289,Morris Brown,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,2024,1302,NE Illinois,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Betting Odds

In [32]:
df_bo = pd.read_parquet('../data/preprocessed/mens_betting/betting.parquet')

df_bo = df_bo.loc[df_bo['Season'] == season, :].reset_index(drop=True)

df_bo

,Season,Team,Implied Champion Probability
0,2024,Connecticut,0.222222
1,2024,Purdue,0.133333
2,2024,Alabama,0.024390
3,2024,NC State,0.004975
4,2024,Tennessee,0.062500
...,...,...,...
63,2024,Montana State,0.000500
64,2024,South Dakota State,0.000500
65,2024,St Peter's,0.000500
66,2024,Stetson,0.000500


In [33]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_bo['Team'].unique())

df_match.head(25)

  0%|          | 0/68 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Connecticut,connecticut,100
1,St Mary's,st. mary's,100
2,Wisconsin,wisconsin,100
3,Florida Atlantic,florida atlantic,100
4,Texas Tech,texas tech,100
5,Mississippi State,mississippi state,100
6,New Mexico,new mexico,100
7,TCU,tcu,100
8,Nebraska,nebraska,100
9,Nevada,nevada,100


In [34]:
df_bo.insert(1, 'TeamID', df_bo['Team'].map(team_to_spelling).map(spelling_to_id))

df_bo

,Season,TeamID,Team,Implied Champion Probability
0,2024,1163,Connecticut,0.222222
1,2024,1345,Purdue,0.133333
2,2024,1104,Alabama,0.024390
3,2024,1301,NC State,0.004975
4,2024,1397,Tennessee,0.062500
...,...,...,...,...
63,2024,1286,Montana State,0.000500
64,2024,1355,South Dakota State,0.000500
65,2024,1389,St Peter's,0.000500
66,2024,1391,Stetson,0.000500


In [35]:
df = pd.merge(
    df,
    df_bo.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875,0.024283,-0.045249,1.003736,1.048985,69.446555,0.063427,23.846923,11.768719,NaN
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075,-0.234301,-0.084381,1.049157,1.133538,63.080511,-0.191559,16.689482,3.046389,NaN
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844,0.648,5.612,0.64900,5.65250,0.457533,0.053016,1.061068,1.008052,67.044145,0.234113,31.015618,17.953874,0.000999
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310,0.955,27.053,0.88425,19.62150,2.258407,0.228423,1.234926,1.006503,73.746625,0.283066,40.127890,28.383921,0.024390
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700,-0.879836,-0.137497,0.935035,1.072533,71.310687,-0.074870,15.888827,3.620456,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.112391,-0.217351,0.914092,1.131443,68.999473,-0.400817,2.053793,-11.474784,NaN
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.182077,-0.167152,0.935817,1.102969,67.356915,-0.111816,12.329532,-0.371308,NaN
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.158761,-0.089166,1.000531,1.089697,68.308798,-0.070744,16.213913,3.887557,NaN
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
df.loc[df['Implied Champion Probability'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875,0.024283,-0.045249,1.003736,1.048985,69.446555,0.063427,23.846923,11.768719,NaN
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075,-0.234301,-0.084381,1.049157,1.133538,63.080511,-0.191559,16.689482,3.046389,NaN
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700,-0.879836,-0.137497,0.935035,1.072533,71.310687,-0.074870,15.888827,3.620456,NaN
5,2024,1106,Alabama St,-1.0,-1.000000,92.6,104.3,-11.7,0.203,0.344828,41.0,49.1,28.5,43.4,16.3,20.0,31.7,29.1,69.1,47.6,34.1,11.1,12.5,44.4,53.3,38.4,41.0,68.1,80.237,1.891,0.402,71.9,13.689,0.100,-19.276,0.10300,-19.29750,-0.891187,-0.118667,0.915038,1.033705,68.995056,-0.195720,10.343909,-2.510459,NaN
6,2024,1107,SUNY Albany,-1.0,-1.000000,103.6,110.2,-6.6,0.330,0.387097,50.4,51.4,29.2,38.9,17.7,17.0,30.6,29.4,74.4,54.6,29.3,7.4,11.5,44.5,46.6,36.9,29.6,74.0,78.852,1.711,0.344,72.5,11.654,0.123,-18.178,0.23875,-10.81025,-0.529850,-0.043991,1.047154,1.091145,73.602203,-0.153456,17.442745,5.047938,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.112391,-0.217351,0.914092,1.131443,68.999473,-0.400817,2.053793,-11.474784,NaN
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.182077,-0.167152,0.935817,1.102969,67.356915,-0.111816,12.329532,-0.371308,NaN
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.158761,-0.089166,1.000531,1.089697,68.308798,-0.070744,16.213913,3.887557,NaN
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Map to Matchups

In [37]:
# df_seeds = pd.read_csv(fr'..\data\unprocessed\kaggle\{season}_tourney_seeds.csv')

# df_seeds = df_seeds.loc[df_seeds['Tournament'] == 'M', :].reset_index(drop=True)

# df_seeds.rename(columns={'Seed': 'Region Seed'}, inplace=True)
# df_seeds.insert(2, 'Region', df_seeds['Region Seed'].str[0])
# df_seeds.insert(3, 'Seed', df_seeds['Region Seed'].str.extract('(\d+)').astype(int))

# df_seeds

In [38]:
df_seeds = pd.read_csv(r'..\data\unprocessed\kaggle\MNCAATourneySeeds.csv')

df_seeds = df_seeds.loc[df_seeds['Season'] == season, :].reset_index(drop=True)

df_seeds.insert(2, 'Play In', df_seeds['Seed'].str.endswith(('a', 'b')))
df_seeds.insert(2, 'Region', df_seeds['Seed'].str[0])
df_seeds['Seed'] = df_seeds['Seed'].str.extract('(\d+)').astype(int)

# df_seeds = df_seeds.loc[~df_seeds['TeamID'].isin(playin_losers), :].reset_index(drop=True)

df_seeds

,Season,Seed,Region,Play In,TeamID
0,2024,1,W,False,1163
1,2024,2,W,False,1235
2,2024,3,W,False,1228
3,2024,4,W,False,1120
4,2024,5,W,False,1361
...,...,...,...,...,...
63,2024,12,Z,False,1241
64,2024,13,Z,False,1436
65,2024,14,Z,False,1324
66,2024,15,Z,False,1443


In [39]:
df = df.merge(
    df_seeds,
    how='left',
    on=['Season', 'TeamID'],
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability,Seed,Region,Play In
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875,0.024283,-0.045249,1.003736,1.048985,69.446555,0.063427,23.846923,11.768719,NaN,NaN,NaN,NaN
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075,-0.234301,-0.084381,1.049157,1.133538,63.080511,-0.191559,16.689482,3.046389,NaN,NaN,NaN,NaN
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844,0.648,5.612,0.64900,5.65250,0.457533,0.053016,1.061068,1.008052,67.044145,0.234113,31.015618,17.953874,0.000999,14.0,Y,False
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310,0.955,27.053,0.88425,19.62150,2.258407,0.228423,1.234926,1.006503,73.746625,0.283066,40.127890,28.383921,0.024390,4.0,X,False
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700,-0.879836,-0.137497,0.935035,1.072533,71.310687,-0.074870,15.888827,3.620456,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.112391,-0.217351,0.914092,1.131443,68.999473,-0.400817,2.053793,-11.474784,NaN,NaN,NaN,NaN
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.182077,-0.167152,0.935817,1.102969,67.356915,-0.111816,12.329532,-0.371308,NaN,NaN,NaN,NaN
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.158761,-0.089166,1.000531,1.089697,68.308798,-0.070744,16.213913,3.887557,NaN,NaN,NaN,NaN
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
id_to_region = dict(zip(df['TeamID'], df['Region']))
id_to_seed = dict(zip(df['TeamID'], df['Seed']))

df_mod = pd.DataFrame(
    [
        (team_a, team_b) 
        for team_a in df['TeamID'].unique() 
        for team_b in df['TeamID'].unique() 
        if team_a != team_b
    ],
    columns=['Team A ID', 'Team B ID']
)

df_mod.insert(0, 'Season', season)
df_mod['Team A Region'] = df_mod['Team A ID'].map(id_to_region)
df_mod['Team B Region'] = df_mod['Team B ID'].map(id_to_region)
df_mod['Team A Seed'] = df_mod['Team A ID'].map(id_to_seed)
df_mod['Team B Seed'] = df_mod['Team B ID'].map(id_to_seed)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed
0,2024,1101,1102,NaN,NaN,NaN,NaN
1,2024,1101,1103,NaN,Y,NaN,14.0
2,2024,1101,1104,NaN,X,NaN,4.0
3,2024,1101,1105,NaN,NaN,NaN,NaN
4,2024,1101,1106,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
144015,2024,1480,1475,NaN,NaN,NaN,NaN
144016,2024,1480,1476,NaN,NaN,NaN,NaN
144017,2024,1480,1477,NaN,NaN,NaN,NaN
144018,2024,1480,1478,NaN,NaN,NaN,NaN


Calculate round of matchup

In [41]:
same_region = df_mod['Team A Region'] == df_mod['Team B Region']

# round_0_condition = (df_mod['team0_playin'] == 1) & (df_mod['team1_playin'] == 1)  # no play-in games in this data

round_1_condition = df_mod['Team A Seed'] + df_mod['Team B Seed'] == 17

round_2_condition = (
    (df_mod['Team A Seed'].isin([1, 16]) & df_mod['Team B Seed'].isin([8, 9])) | 
    (df_mod['Team A Seed'].isin([8, 9]) & df_mod['Team B Seed'].isin([1, 16])) |
    (df_mod['Team A Seed'].isin([5, 12]) & df_mod['Team B Seed'].isin([4, 13])) | 
    (df_mod['Team A Seed'].isin([4, 13]) & df_mod['Team B Seed'].isin([5, 12])) |
    (df_mod['Team A Seed'].isin([6, 11]) & df_mod['Team B Seed'].isin([3, 14])) | 
    (df_mod['Team A Seed'].isin([3, 14]) & df_mod['Team B Seed'].isin([6, 11])) |
    (df_mod['Team A Seed'].isin([7, 10]) & df_mod['Team B Seed'].isin([2, 15])) | 
    (df_mod['Team A Seed'].isin([2, 15]) & df_mod['Team B Seed'].isin([7, 10]))
)

round_3_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9]) & df_mod['Team B Seed'].isin([5, 12, 4, 13])) | 
    (df_mod['Team A Seed'].isin([5, 12, 4, 13]) & df_mod['Team B Seed'].isin([1, 16, 8, 9])) |
    (df_mod['Team A Seed'].isin([6, 11, 3, 14]) & df_mod['Team B Seed'].isin([7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([7, 10, 2, 15]) & df_mod['Team B Seed'].isin([6, 11, 3, 14]))
)

round_4_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]) & df_mod['Team B Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15]) & df_mod['Team B Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]))
)

round_5_condition = (
    (df_mod['Team A Region'].isin(['W']) & df_mod['Team B Region'].isin(['X'])) | 
    (df_mod['Team A Region'].isin(['X']) & df_mod['Team B Region'].isin(['W'])) |
    (df_mod['Team A Region'].isin(['Y']) & df_mod['Team B Region'].isin(['Z'])) | 
    (df_mod['Team A Region'].isin(['Z']) & df_mod['Team B Region'].isin(['Y']))
)

round_6_condition = (
    (df_mod['Team A Region'].isin(['W', 'X']) & df_mod['Team B Region'].isin(['Y', 'Z'])) | 
    (df_mod['Team A Region'].isin(['Y', 'Z']) & df_mod['Team B Region'].isin(['W', 'X'])) 
)

round_6_condition

0         False
1         False
2         False
3         False
4         False
          ...  
144015    False
144016    False
144017    False
144018    False
144019    False
Length: 144020, dtype: bool

In [42]:
df_mod['Round'] = float('nan')

df_mod.loc[round_6_condition, 'Round'] = 6

df_mod.loc[round_5_condition, 'Round'] = 5

df_mod.loc[round_4_condition & same_region, 'Round'] = 4

df_mod.loc[round_3_condition & same_region, 'Round'] = 3

df_mod.loc[round_2_condition & same_region, 'Round'] = 2

df_mod.loc[round_1_condition & same_region, 'Round'] = 1

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round
0,2024,1101,1102,NaN,NaN,NaN,NaN,NaN
1,2024,1101,1103,NaN,Y,NaN,14.0,NaN
2,2024,1101,1104,NaN,X,NaN,4.0,NaN
3,2024,1101,1105,NaN,NaN,NaN,NaN,NaN
4,2024,1101,1106,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
144015,2024,1480,1475,NaN,NaN,NaN,NaN,NaN
144016,2024,1480,1476,NaN,NaN,NaN,NaN,NaN
144017,2024,1480,1477,NaN,NaN,NaN,NaN,NaN
144018,2024,1480,1478,NaN,NaN,NaN,NaN,NaN


Get Head-to-Head

In [43]:
df_h2h = pd.read_parquet('../data/preprocessed/mens_h2h/h2h.parquet')

df_h2h = df_h2h.loc[df_h2h['Season'] == season, :].reset_index(drop=True)

df_h2h

,Season,Team A,Team B,Head to Head,Common Opps
0,2024,Abilene Christian,Air Force,NaN,0.960669
1,2024,Abilene Christian,Alabama,NaN,-1.276758
2,2024,Abilene Christian,Alabama A&M,NaN,0.117263
3,2024,Abilene Christian,Alabama State,NaN,-1.253728
4,2024,Abilene Christian,Alcorn State,NaN,-0.187852
...,...,...,...,...,...
74107,2024,Youngstown State,William & Mary,NaN,1.438673
74108,2024,Youngstown State,Winthrop,NaN,0.006061
74109,2024,Youngstown State,Wisconsin,NaN,-0.111978
74110,2024,Youngstown State,Wright State,0.791622,-0.187724


In [44]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_h2h['Team A'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Abilene Christian,abilene christian,100
1,Quinnipiac,quinnipiac,100
2,Queens (NC),Queens (NC),100
3,Purdue Fort Wayne,purdue fort wayne,100
4,Purdue,purdue,100
5,Providence,providence,100
6,Princeton,princeton,100
7,Presbyterian,presbyterian,100
8,Prairie View,prairie view,100
9,Radford,radford,100


In [45]:
df_h2h.insert(df_h2h.columns.get_loc('Team A'), 'Team A ID', df_h2h['Team A'].map(team_to_spelling).map(spelling_to_id))

df_h2h.insert(df_h2h.columns.get_loc('Team B'), 'Team B ID', df_h2h['Team B'].map(team_to_spelling).map(spelling_to_id))

df_h2h

,Season,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps
0,2024,1101,Abilene Christian,1102,Air Force,NaN,0.960669
1,2024,1101,Abilene Christian,1104,Alabama,NaN,-1.276758
2,2024,1101,Abilene Christian,1105,Alabama A&M,NaN,0.117263
3,2024,1101,Abilene Christian,1106,Alabama State,NaN,-1.253728
4,2024,1101,Abilene Christian,1108,Alcorn State,NaN,-0.187852
...,...,...,...,...,...,...,...
74107,2024,1464,Youngstown State,1456,William & Mary,NaN,1.438673
74108,2024,1464,Youngstown State,1457,Winthrop,NaN,0.006061
74109,2024,1464,Youngstown State,1458,Wisconsin,NaN,-0.111978
74110,2024,1464,Youngstown State,1460,Wright State,0.791622,-0.187724


In [46]:
df_mod = pd.merge(
    df_mod,
    df_h2h[['Season', 'Team A ID', 'Team B ID', 'Head to Head', 'Common Opps']],
    how='left',
    on=['Season', 'Team A ID', 'Team B ID'],
)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Head to Head,Common Opps
0,2024,1101,1102,NaN,NaN,NaN,NaN,NaN,NaN,0.960669
1,2024,1101,1103,NaN,Y,NaN,14.0,NaN,NaN,NaN
2,2024,1101,1104,NaN,X,NaN,4.0,NaN,NaN,-1.276758
3,2024,1101,1105,NaN,NaN,NaN,NaN,NaN,NaN,0.117263
4,2024,1101,1106,NaN,NaN,NaN,NaN,NaN,NaN,-1.253728
...,...,...,...,...,...,...,...,...,...,...
144015,2024,1480,1475,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144016,2024,1480,1476,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144017,2024,1480,1477,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144018,2024,1480,1478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Get team names

In [47]:
df_teams = pd.read_csv(r'..\data\unprocessed\kaggle\MTeams.csv')

df_teams

,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2025
1,1102,Air Force,1985,2025
2,1103,Akron,1985,2025
3,1104,Alabama,1985,2025
4,1105,Alabama A&M,2000,2025
...,...,...,...,...
375,1476,Stonehill,2023,2025
376,1477,East Texas A&M,2023,2025
377,1478,Le Moyne,2024,2025
378,1479,Mercyhurst,2025,2025


In [48]:
id_to_team = dict(zip(df_teams['TeamID'], df_teams['TeamName']))

df_mod.insert(df_mod.columns.get_loc('Team A ID') + 1, 'Team A', df_mod['Team A ID'].map(id_to_team))
df_mod.insert(df_mod.columns.get_loc('Team B ID') + 1, 'Team B', df_mod['Team B ID'].map(id_to_team))

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Head to Head,Common Opps
0,2024,1101,Abilene Chr,1102,Air Force,NaN,NaN,NaN,NaN,NaN,NaN,0.960669
1,2024,1101,Abilene Chr,1103,Akron,NaN,Y,NaN,14.0,NaN,NaN,NaN
2,2024,1101,Abilene Chr,1104,Alabama,NaN,X,NaN,4.0,NaN,NaN,-1.276758
3,2024,1101,Abilene Chr,1105,Alabama A&M,NaN,NaN,NaN,NaN,NaN,NaN,0.117263
4,2024,1101,Abilene Chr,1106,Alabama St,NaN,NaN,NaN,NaN,NaN,NaN,-1.253728
...,...,...,...,...,...,...,...,...,...,...,...,...
144015,2024,1480,West Georgia,1475,Southern Indiana,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144016,2024,1480,West Georgia,1476,Stonehill,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144017,2024,1480,West Georgia,1477,East Texas A&M,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144018,2024,1480,West Georgia,1478,Le Moyne,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Map features

In [49]:
team_a_features = pd.merge(
    df_mod[['Season', 'Team A ID']],
    df.drop(columns=['Team', 'Seed', 'Region', 'Play In']),
    how='left',
    left_on=['Season', 'Team A ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team A ID', 'TeamID'])

team_b_features = pd.merge(
    df_mod[['Season', 'Team B ID']],
    df.drop(columns=['Team', 'Seed', 'Region', 'Play In']),
    how='left',
    left_on=['Season', 'Team B ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team B ID', 'TeamID'])

df_features = team_a_features - team_b_features

df_features['Team A ADJ OE Team B ADJ DE'] = team_a_features['ADJ OE'] + team_b_features['ADJ DE']
df_features['Team B ADJ OE Team A ADJ DE'] = team_b_features['ADJ OE'] + team_a_features['ADJ DE']

df_features['Team A Offense Team B Defense'] = team_a_features['Adjusted Offense'] + team_b_features['Adjusted Defense']
df_features['Team B Offense Team A Defense'] = team_b_features['Adjusted Offense'] + team_a_features['Adjusted Defense']

df_features['Team A BARTHAG'] = team_a_features['BARTHAG']
df_features['Team B BARTHAG'] = team_b_features['BARTHAG']

df_features

,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,0.0,0.666667,-4.6,-6.4,1.8,0.041,0.164223,-6.7,-2.6,8.3,0.8,-1.1,2.6,2.1,-1.1,7.2,-0.7,-4.1,-6.1,-3.3,-13.4,-2.7,-20.8,-5.1,6.5,-0.023,0.593,4.692,4.9,-6.301,-0.125,-4.561,0.21775,8.30950,0.258584,0.039132,-0.045420,-0.084553,6.366044,0.254986,7.157441,8.722330,NaN,212.0,210.2,2.137274,2.098142,0.401,0.360
1,0.0,0.333333,-4.7,2.7,-7.4,-0.203,-0.232955,-4.9,3.0,4.3,10.9,0.5,3.6,-3.6,4.8,3.9,2.3,2.5,-2.3,0.2,0.4,1.8,-15.1,-5.2,2.7,-0.995,-0.419,-6.108,0.7,1.928,-0.206,-7.749,-0.06025,-2.49375,-0.433250,-0.098264,-0.057331,0.040933,2.402410,-0.170686,-7.168695,-6.185154,NaN,202.9,210.3,2.011788,2.110053,0.401,0.604
2,-3.0,-1.666667,-24.6,3.2,-27.8,-0.520,-0.201705,-9.2,1.7,2.7,0.3,1.7,4.6,-9.0,0.5,-3.6,1.9,0.6,-4.5,-4.5,-2.9,2.1,-20.4,-5.1,-4.1,-3.483,0.061,-22.878,-5.3,-22.538,-0.513,-29.190,-0.29550,-16.46275,-2.234124,-0.273672,-0.231190,0.042482,-4.300070,-0.219639,-16.280967,-16.615201,NaN,202.4,230.2,2.010239,2.283911,0.401,0.921
3,0.0,0.666667,7.9,-4.3,12.2,0.259,0.101604,1.3,2.7,-9.8,-6.0,-4.8,0.6,-6.0,-2.0,-1.3,6.3,-2.6,-5.2,-0.3,3.3,-6.6,-0.3,-5.5,-2.1,-0.804,0.285,4.692,1.1,3.854,0.230,9.449,0.45875,19.83575,0.904119,0.092249,0.068701,-0.023547,-1.864132,0.138297,7.958096,8.148263,NaN,209.9,197.7,2.076269,1.984020,0.401,0.142
4,0.0,0.666667,8.4,0.3,8.1,0.198,0.109718,6.1,2.5,9.4,-3.5,1.4,0.2,-5.8,1.3,1.5,5.4,-1.6,-5.4,-4.5,4.2,-6.1,-12.0,-9.4,1.3,-1.260,0.272,4.490,1.2,0.083,0.342,17.139,0.48575,22.45625,0.915470,0.073418,0.088699,0.015280,0.451499,0.259147,13.503014,14.279178,NaN,205.3,197.2,2.037441,1.964023,0.401,0.203
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144394,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.158
144395,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.068
144396,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.128
144397,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.206


In [50]:
df_mod[df_features.columns] = df_features

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,2024,1101,Abilene Chr,1102,Air Force,NaN,NaN,NaN,NaN,NaN,NaN,0.960669,0.0,0.666667,-4.6,-6.4,1.8,0.041,0.164223,-6.7,-2.6,8.3,0.8,-1.1,2.6,2.1,-1.1,7.2,-0.7,-4.1,-6.1,-3.3,-13.4,-2.7,-20.8,-5.1,6.5,-0.023,0.593,4.692,4.9,-6.301,-0.125,-4.561,0.21775,8.30950,0.258584,0.039132,-0.045420,-0.084553,6.366044,0.254986,7.157441,8.722330,NaN,212.0,210.2,2.137274,2.098142,0.401,0.360
1,2024,1101,Abilene Chr,1103,Akron,NaN,Y,NaN,14.0,NaN,NaN,NaN,0.0,0.333333,-4.7,2.7,-7.4,-0.203,-0.232955,-4.9,3.0,4.3,10.9,0.5,3.6,-3.6,4.8,3.9,2.3,2.5,-2.3,0.2,0.4,1.8,-15.1,-5.2,2.7,-0.995,-0.419,-6.108,0.7,1.928,-0.206,-7.749,-0.06025,-2.49375,-0.433250,-0.098264,-0.057331,0.040933,2.402410,-0.170686,-7.168695,-6.185154,NaN,202.9,210.3,2.011788,2.110053,0.401,0.604
2,2024,1101,Abilene Chr,1104,Alabama,NaN,X,NaN,4.0,NaN,NaN,-1.276758,-3.0,-1.666667,-24.6,3.2,-27.8,-0.520,-0.201705,-9.2,1.7,2.7,0.3,1.7,4.6,-9.0,0.5,-3.6,1.9,0.6,-4.5,-4.5,-2.9,2.1,-20.4,-5.1,-4.1,-3.483,0.061,-22.878,-5.3,-22.538,-0.513,-29.190,-0.29550,-16.46275,-2.234124,-0.273672,-0.231190,0.042482,-4.300070,-0.219639,-16.280967,-16.615201,NaN,202.4,230.2,2.010239,2.283911,0.401,0.921
3,2024,1101,Abilene Chr,1105,Alabama A&M,NaN,NaN,NaN,NaN,NaN,NaN,0.117263,0.0,0.666667,7.9,-4.3,12.2,0.259,0.101604,1.3,2.7,-9.8,-6.0,-4.8,0.6,-6.0,-2.0,-1.3,6.3,-2.6,-5.2,-0.3,3.3,-6.6,-0.3,-5.5,-2.1,-0.804,0.285,4.692,1.1,3.854,0.230,9.449,0.45875,19.83575,0.904119,0.092249,0.068701,-0.023547,-1.864132,0.138297,7.958096,8.148263,NaN,209.9,197.7,2.076269,1.984020,0.401,0.142
4,2024,1101,Abilene Chr,1106,Alabama St,NaN,NaN,NaN,NaN,NaN,NaN,-1.253728,0.0,0.666667,8.4,0.3,8.1,0.198,0.109718,6.1,2.5,9.4,-3.5,1.4,0.2,-5.8,1.3,1.5,5.4,-1.6,-5.4,-4.5,4.2,-6.1,-12.0,-9.4,1.3,-1.260,0.272,4.490,1.2,0.083,0.342,17.139,0.48575,22.45625,0.915470,0.073418,0.088699,0.015280,0.451499,0.259147,13.503014,14.279178,NaN,205.3,197.2,2.037441,1.964023,0.401,0.203
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144015,2024,1480,West Georgia,1475,Southern Indiana,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.068
144016,2024,1480,West Georgia,1476,Stonehill,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.128
144017,2024,1480,West Georgia,1477,East Texas A&M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.206
144018,2024,1480,West Georgia,1478,Le Moyne,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
# track if game is a tournament matchup for later
tournament_matchup = (df_mod['Team A Region'].notna()) & (df_mod['Team B Region'].notna())

tournament_matchup.sum()

4556

In [52]:
df_mod.insert(1, 'Round', df_mod.pop('Round'))

df_mod.drop(columns=['Team A Region', 'Team B Region', 'Team A Seed', 'Team B Seed'], inplace=True)

df_mod

,Season,Round,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,2024,NaN,1101,Abilene Chr,1102,Air Force,NaN,0.960669,0.0,0.666667,-4.6,-6.4,1.8,0.041,0.164223,-6.7,-2.6,8.3,0.8,-1.1,2.6,2.1,-1.1,7.2,-0.7,-4.1,-6.1,-3.3,-13.4,-2.7,-20.8,-5.1,6.5,-0.023,0.593,4.692,4.9,-6.301,-0.125,-4.561,0.21775,8.30950,0.258584,0.039132,-0.045420,-0.084553,6.366044,0.254986,7.157441,8.722330,NaN,212.0,210.2,2.137274,2.098142,0.401,0.360
1,2024,NaN,1101,Abilene Chr,1103,Akron,NaN,NaN,0.0,0.333333,-4.7,2.7,-7.4,-0.203,-0.232955,-4.9,3.0,4.3,10.9,0.5,3.6,-3.6,4.8,3.9,2.3,2.5,-2.3,0.2,0.4,1.8,-15.1,-5.2,2.7,-0.995,-0.419,-6.108,0.7,1.928,-0.206,-7.749,-0.06025,-2.49375,-0.433250,-0.098264,-0.057331,0.040933,2.402410,-0.170686,-7.168695,-6.185154,NaN,202.9,210.3,2.011788,2.110053,0.401,0.604
2,2024,NaN,1101,Abilene Chr,1104,Alabama,NaN,-1.276758,-3.0,-1.666667,-24.6,3.2,-27.8,-0.520,-0.201705,-9.2,1.7,2.7,0.3,1.7,4.6,-9.0,0.5,-3.6,1.9,0.6,-4.5,-4.5,-2.9,2.1,-20.4,-5.1,-4.1,-3.483,0.061,-22.878,-5.3,-22.538,-0.513,-29.190,-0.29550,-16.46275,-2.234124,-0.273672,-0.231190,0.042482,-4.300070,-0.219639,-16.280967,-16.615201,NaN,202.4,230.2,2.010239,2.283911,0.401,0.921
3,2024,NaN,1101,Abilene Chr,1105,Alabama A&M,NaN,0.117263,0.0,0.666667,7.9,-4.3,12.2,0.259,0.101604,1.3,2.7,-9.8,-6.0,-4.8,0.6,-6.0,-2.0,-1.3,6.3,-2.6,-5.2,-0.3,3.3,-6.6,-0.3,-5.5,-2.1,-0.804,0.285,4.692,1.1,3.854,0.230,9.449,0.45875,19.83575,0.904119,0.092249,0.068701,-0.023547,-1.864132,0.138297,7.958096,8.148263,NaN,209.9,197.7,2.076269,1.984020,0.401,0.142
4,2024,NaN,1101,Abilene Chr,1106,Alabama St,NaN,-1.253728,0.0,0.666667,8.4,0.3,8.1,0.198,0.109718,6.1,2.5,9.4,-3.5,1.4,0.2,-5.8,1.3,1.5,5.4,-1.6,-5.4,-4.5,4.2,-6.1,-12.0,-9.4,1.3,-1.260,0.272,4.490,1.2,0.083,0.342,17.139,0.48575,22.45625,0.915470,0.073418,0.088699,0.015280,0.451499,0.259147,13.503014,14.279178,NaN,205.3,197.2,2.037441,1.964023,0.401,0.203
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144015,2024,NaN,1480,West Georgia,1475,Southern Indiana,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.068
144016,2024,NaN,1480,West Georgia,1476,Stonehill,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.128
144017,2024,NaN,1480,West Georgia,1477,East Texas A&M,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.206
144018,2024,NaN,1480,West Georgia,1478,Le Moyne,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Drop features that were not used in the model

In [53]:
df_mod.drop(
    columns=[
        # 'Team A ADJ OE Team B ADJ DE',
        # 'Team B ADJ OE Team A ADJ DE',
        '3P RATE D',
        # 'Past Year ADJ EM',
        'EFF. HGT.',
        'EFG',
        'WIN%',
        # 'Efficiency Margin',
        'EXP.',
        'FT RATE D',
        'Team A Offense Team B Defense',
        'Team B Offense Team A Defense',
        # 'Past Year Tournament Result',
        'Round',
        'ADJ. T',
        'Head to Head',
        'EFG D.',

        # 'Team A BARTHAG',
        # 'Team B BARTHAG',

        # 'Past Year Tournament Result',

        # 'TOV%',
        # 'TOV% D',

        'OP OREB%',

        # 'Mu',
    ],
    inplace=True,
)

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,FT RATE,TOV%,TOV% D,O REB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A BARTHAG,Team B BARTHAG
0,2024,1101,Abilene Chr,1102,Air Force,0.960669,0.0,0.666667,-4.6,-6.4,1.8,0.041,8.3,-1.1,2.6,2.1,7.2,-0.7,-4.1,-6.1,-3.3,-13.4,-2.7,-20.8,4.692,4.9,-6.301,-0.125,-4.561,0.21775,8.30950,0.258584,0.039132,-0.045420,-0.084553,6.366044,0.254986,7.157441,8.722330,NaN,212.0,210.2,0.401,0.360
1,2024,1101,Abilene Chr,1103,Akron,NaN,0.0,0.333333,-4.7,2.7,-7.4,-0.203,4.3,0.5,3.6,-3.6,3.9,2.3,2.5,-2.3,0.2,0.4,1.8,-15.1,-6.108,0.7,1.928,-0.206,-7.749,-0.06025,-2.49375,-0.433250,-0.098264,-0.057331,0.040933,2.402410,-0.170686,-7.168695,-6.185154,NaN,202.9,210.3,0.401,0.604
2,2024,1101,Abilene Chr,1104,Alabama,-1.276758,-3.0,-1.666667,-24.6,3.2,-27.8,-0.520,2.7,1.7,4.6,-9.0,-3.6,1.9,0.6,-4.5,-4.5,-2.9,2.1,-20.4,-22.878,-5.3,-22.538,-0.513,-29.190,-0.29550,-16.46275,-2.234124,-0.273672,-0.231190,0.042482,-4.300070,-0.219639,-16.280967,-16.615201,NaN,202.4,230.2,0.401,0.921
3,2024,1101,Abilene Chr,1105,Alabama A&M,0.117263,0.0,0.666667,7.9,-4.3,12.2,0.259,-9.8,-4.8,0.6,-6.0,-1.3,6.3,-2.6,-5.2,-0.3,3.3,-6.6,-0.3,4.692,1.1,3.854,0.230,9.449,0.45875,19.83575,0.904119,0.092249,0.068701,-0.023547,-1.864132,0.138297,7.958096,8.148263,NaN,209.9,197.7,0.401,0.142
4,2024,1101,Abilene Chr,1106,Alabama St,-1.253728,0.0,0.666667,8.4,0.3,8.1,0.198,9.4,1.4,0.2,-5.8,1.5,5.4,-1.6,-5.4,-4.5,4.2,-6.1,-12.0,4.490,1.2,0.083,0.342,17.139,0.48575,22.45625,0.915470,0.073418,0.088699,0.015280,0.451499,0.259147,13.503014,14.279178,NaN,205.3,197.2,0.401,0.203
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144015,2024,1480,West Georgia,1475,Southern Indiana,NaN,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.068
144016,2024,1480,West Georgia,1476,Stonehill,NaN,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.128
144017,2024,1480,West Georgia,1477,East Texas A&M,NaN,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.206
144018,2024,1480,West Georgia,1478,Le Moyne,NaN,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Check that data follows same format as the data that the model was trained on

In [54]:
df_mod_training = pd.read_parquet(data_path)

assert all(df_mod_training.drop(columns=['Result']).columns == df_mod.columns), 'Columns do not match'

'Columns Match'

'Columns Match'

### Get Model Predictions

In [55]:
import pickle

with open(model_path, 'rb') as f:
    mod = pickle.load(f)

mod

LGBMClassifier(early_stopping_round=25, feature_fraction=0.18250666863795148,
               lambda_l1=1.2272140531521787, lambda_l2=0.41222610005136445,
               learning_rate=0.08417263027166265, max_depth=5, metric='rmse',
               min_child_samples=13,
               monotone_constraints=[1, 1, 1, 1, -1, 1, 1, -1, -1, 0, 1, 0, -1,
                                     -1, 1, -1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1,
                                     1, 1, -1, ...],
               n_estimators=1000, num_leaves=134, random_state=22,
               verbosity=-1)

In [56]:
X = df_mod.drop(columns=['Season', 'Team A ID', 'Team A', 'Team B ID', 'Team B'])

df_mod['Prediction'] = mod.predict_proba(X)[:, 1]

df_mod['Prediction']

0         0.469719
1         0.296314
2         0.101557
3         0.814349
4         0.616167
            ...   
144015    0.535120
144016    0.535120
144017    0.535120
144018    0.535120
144019    0.531121
Name: Prediction, Length: 144020, dtype: float64

In [57]:
(
    df_mod[['Season', 'Team A ID', 'Team A', 'Team B ID', 'Team B', 'Prediction']]
    .to_parquet(f'../data/simulations/mens/matchup_predictions_{season}.parquet')
)

'Done'

'Done'

Turn predictions into matchup matrix

In [58]:
# filter down to just tournament games
df_mod = df_mod.loc[tournament_matchup, :].reset_index(drop=True)

# filter out play-in losers
df_mod = df_mod.loc[
    (~df_mod['Team A ID'].isin(playin_losers)) & (~df_mod['Team B ID'].isin(playin_losers)), 
    :
].reset_index(drop=True)

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,FT RATE,TOV%,TOV% D,O REB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A BARTHAG,Team B BARTHAG,Prediction
0,2024,1103,Akron,1104,Alabama,NaN,0.0,-0.333333,4.7,-2.7,7.4,0.203,-4.3,-0.5,-3.6,3.6,-3.9,-2.3,-2.5,2.3,-0.2,-0.4,-1.8,15.1,6.108,-0.7,-1.928,0.206,7.749,0.06025,2.493750,0.433250,0.098264,0.057331,-0.040933,-2.402410,0.170686,7.168695,6.185154,NaN,210.3,202.9,0.604,0.401,0.681335
1,2024,1103,Akron,1112,Arizona,NaN,0.0,0.333333,2.5,-11.7,14.2,0.355,5.2,-0.1,-0.4,1.6,3.8,-1.2,-6.9,2.5,-0.7,-8.2,-6.5,-3.4,10.800,0.0,3.497,0.390,15.247,0.37225,14.872250,1.651654,0.137384,0.023278,-0.114106,3.142059,0.336278,14.483873,13.997705,NaN,219.3,205.1,0.604,0.249,0.831049
2,2024,1103,Akron,1120,Auburn,-1.190417,0.0,0.333333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.604,NaN,0.523321
3,2024,1103,Akron,1124,Baylor,-0.197963,0.0,0.333333,0.0,-11.4,11.4,0.294,1.7,3.9,0.0,-0.4,1.1,-3.7,-4.0,0.8,-1.4,8.4,-0.5,6.4,10.608,1.1,2.073,0.487,20.364,0.31000,12.224250,0.710617,0.108908,-0.005660,-0.114568,0.965316,0.152909,9.003774,8.229862,NaN,219.0,207.6,0.604,0.310,0.696565
4,2024,1103,Akron,1140,BYU,NaN,0.0,0.333333,10.7,-12.6,23.3,0.499,7.6,-2.8,1.5,1.2,-3.4,-2.8,-6.5,-3.0,-0.3,-4.2,-8.2,0.1,3.177,1.3,0.422,0.195,7.305,0.06425,2.350500,2.126742,0.236432,0.115574,-0.120858,-3.232398,0.656480,27.379145,28.211578,NaN,220.2,196.9,0.604,0.105,0.805472
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4027,2024,1463,Yale,1436,Vermont,0.499725,3.0,1.000000,5.6,-14.8,20.4,0.487,-2.2,-1.5,0.5,-4.1,1.0,-1.0,-0.8,-2.9,-0.1,9.5,4.1,-6.8,37.690,0.9,26.957,0.542,25.257,0.56275,24.068250,1.062176,0.145706,0.028114,-0.117593,1.122416,0.070210,4.943642,5.270661,NaN,225.0,204.6,0.816,0.329,0.780264
4028,2024,1463,Yale,1443,WKU,NaN,3.0,0.666667,4.5,-8.6,13.1,0.303,-6.2,-2.1,0.1,3.3,3.8,-4.5,-1.0,1.7,0.0,13.3,10.1,-5.0,31.042,-0.3,13.846,0.353,18.205,0.29475,13.134250,0.461236,0.104291,0.027612,-0.076679,3.712479,0.127047,4.736031,5.485710,NaN,218.8,205.7,0.816,0.513,0.824311
4029,2024,1463,Yale,1447,Wagner,0.229163,3.0,1.000000,8.0,-13.9,21.9,0.524,-1.1,1.8,1.8,4.9,2.9,-1.4,-2.0,0.3,2.0,19.1,12.6,-2.6,35.331,-2.6,26.198,0.570,26.514,0.59425,25.040750,1.664416,0.169924,0.047840,-0.122084,3.579131,0.164380,11.176739,11.955224,NaN,224.1,202.2,0.816,0.292,0.914657
4030,2024,1463,Yale,1450,Washington St,-0.016555,3.0,1.000000,12.8,-9.1,21.9,0.532,-8.5,-5.4,-2.3,3.7,-0.2,-0.9,-0.5,-3.2,1.2,7.5,7.0,-2.7,31.640,10.9,23.116,0.314,16.748,0.49750,21.369167,1.711203,0.199707,0.107033,-0.092674,1.020212,0.289306,14.250775,15.108870,NaN,219.3,197.4,0.816,0.284,0.898369


In [59]:
df_matrix = (
    df_mod[['Team A ID', 'Team B ID', 'Prediction']]
    .pivot(
        index=['Team A ID'], 
        columns=['Team B ID'],
        values='Prediction',
    )
)

df_matrix

Team B ID,1103,1104,1112,1120,1124,1140,1155,1158,1159,1160,1161,1163,1166,1173,1179,1181,1182,1194,1196,1211,1212,1213,1222,1228,1235,1241,1242,1246,1253,1255,1266,1270,1277,1280,1287,1301,1304,1305,1307,1314,1321,1324,1332,1345,1355,1359,1361,1376,1388,1389,1391,1395,1397,1400,1401,1403,1412,1429,1436,1443,1447,1450,1458,1463
Team A ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1103,NaN,0.681335,0.831049,0.523321,0.696565,0.805472,0.112034,0.570833,0.894647,0.402877,0.644913,0.151192,0.887894,0.881662,0.838925,0.309925,0.461894,0.904588,0.245699,0.702399,0.423979,0.082061,0.492097,0.807089,0.748596,0.799452,0.670907,0.794294,0.523266,0.628967,0.900818,0.279634,0.823672,0.377138,0.495613,0.879640,0.535120,0.773118,0.156769,0.878447,0.894599,0.898542,0.815797,0.362083,0.433047,0.817320,0.500156,0.406468,0.107899,0.495394,0.145020,0.608270,0.628759,0.050154,0.811104,0.150794,0.709258,0.474367,0.413944,0.821007,0.847350,0.811004,0.551445,0.557967
1104,0.601692,NaN,0.596533,0.897719,0.575449,0.925010,0.920218,0.652409,0.891325,0.927924,0.794895,0.597610,0.207305,0.935670,0.928903,0.927479,0.665766,0.916319,0.893144,0.749766,0.875876,0.847106,0.875418,0.922037,0.831078,0.936753,0.927775,0.699666,0.946944,0.893193,0.896685,0.938126,0.746026,0.692793,0.889829,0.935115,0.735181,0.601692,0.585830,0.874495,0.891450,0.724630,0.818651,0.933612,0.929505,0.860040,0.859452,0.917123,0.716937,0.544539,0.902214,0.897274,0.759024,0.888558,0.589987,0.934561,0.910678,0.762653,0.635642,0.730004,0.851004,0.596533,0.905578,0.885412
1112,0.922879,0.933834,NaN,0.928383,0.892733,0.834536,0.925722,0.597160,0.945849,0.927109,0.918501,0.915448,0.610604,0.931804,0.934323,0.943797,0.940432,0.936112,0.925560,0.917723,0.912929,0.925992,0.594232,0.922798,0.953821,0.936891,0.873405,0.519624,0.733738,0.951831,0.945251,0.943383,0.619970,0.895777,0.835505,0.940724,0.915860,0.909786,0.928382,0.934315,0.936855,0.935679,0.746451,0.948466,0.849900,0.933046,0.877886,0.932748,0.941347,0.904211,0.930512,0.851330,0.696166,0.615953,0.911441,0.916747,0.678123,0.893897,0.847945,0.933275,0.903548,0.789422,0.578321,0.932375
1120,0.933028,0.940752,0.929295,NaN,0.496611,0.578321,0.930984,0.816764,0.690092,0.916244,0.937292,0.924730,0.578321,0.927707,0.769612,0.922996,0.360884,0.928005,0.892822,0.935089,0.932682,0.864667,0.920529,0.915464,0.636116,0.301716,0.933690,0.910094,0.807272,0.925036,0.908072,0.931171,0.905217,0.875886,0.855829,0.939503,0.915451,0.933407,0.935852,0.923673,0.597160,0.864556,0.926405,0.806695,0.910931,0.934462,0.947476,0.932238,0.936554,0.947536,0.935191,0.793084,0.902097,0.934623,0.908914,0.795227,0.907090,0.942743,0.868408,0.919991,0.932493,0.943243,0.826833,0.921485
1124,0.878751,0.898735,0.909675,0.933393,NaN,0.702321,0.946633,0.924934,0.845732,0.904171,0.923188,0.686668,0.798367,0.836096,0.894411,0.771460,0.889688,0.939786,0.638798,0.930735,0.943411,0.898221,0.737829,0.924832,0.928374,0.931961,0.905987,0.940718,0.948973,0.732032,0.726202,0.900689,0.923243,0.861375,0.908850,0.782912,0.838844,0.915598,0.909759,0.917102,0.917629,0.630623,0.918458,0.928519,0.676640,0.714193,0.642416,0.814790,0.840721,0.618887,0.936332,0.910338,0.628577,0.907512,0.754244,0.912345,0.533875,0.855797,0.896282,0.946125,0.913826,0.861517,0.931307,0.704228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1443,0.716640,0.294086,0.414739,0.307633,0.047961,0.106475,0.148439,0.641298,0.509095,0.101202,0.334425,0.402049,0.355461,0.472572,0.374268,0.185214,0.660055,0.455108,0.078758,0.541764,0.240296,0.416343,0.110543,0.347991,0.140394,0.102277,0.254634,0.426042,0.463577,0.432667,0.121619,0.668776,0.532743,0.628083,0.092202,0.440583,0.204741,0.083276,0.385779,0.209030,0.471674,0.444169,0.108370,0.745073,0.112435,0.063490,0.771696,0.134257,0.317367,0.483210,0

In [60]:
df_matrix_display = df_matrix.copy()

df_matrix_display.columns = df_matrix_display.columns.map(id_to_team)
df_matrix_display.index = df_matrix_display.index.map(id_to_team)

df_matrix_display

Team B ID,Akron,Alabama,Arizona,Auburn,Baylor,BYU,Clemson,Col Charleston,Colgate,Colorado,Colorado St,Connecticut,Creighton,Dayton,Drake,Duke,Duquesne,FL Atlantic,Florida,Gonzaga,Grambling,Grand Canyon,Houston,Illinois,Iowa St,James Madison,Kansas,Kentucky,Long Beach St,Longwood,Marquette,McNeese St,Michigan St,Mississippi St,Morehead St,NC State,Nebraska,Nevada,New Mexico,North Carolina,Northwestern,Oakland,Oregon,Purdue,S Dakota St,Samford,San Diego St,South Carolina,St Mary's CA,St Peter's,Stetson,TCU,Tennessee,Texas,Texas A&M,Texas Tech,UAB,Utah St,Vermont,WKU,Wagner,Washington St,Wisconsin,Yale
Team A ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Akron,NaN,0.681335,0.831049,0.523321,0.696565,0.805472,0.112034,0.570833,0.894647,0.402877,0.644913,0.151192,0.887894,0.881662,0.838925,0.309925,0.461894,0.904588,0.245699,0.702399,0.423979,0.082061,0.492097,0.807089,0.748596,0.799452,0.670907,0.794294,0.523266,0.628967,0.900818,0.279634,0.823672,0.377138,0.495613,0.879640,0.535120,0.773118,0.156769,0.878447,0.894599,0.898542,0.815797,0.362083,0.433047,0.817320,0.500156,0.406468,0.107899,0.495394,0.145020,0.608270,0.628759,0.050154,0.811104,0.150794,0.709258,0.474367,0.413944,0.821007,0.847350,0.811004,0.551445,0.557967
Alabama,0.601692,NaN,0.596533,0.897719,0.575449,0.925010,0.920218,0.652409,0.891325,0.927924,0.794895,0.597610,0.207305,0.935670,0.928903,0.927479,0.665766,0.916319,0.893144,0.749766,0.875876,0.847106,0.875418,0.922037,0.831078,0.936753,0.927775,0.699666,0.946944,0.893193,0.896685,0.938126,0.746026,0.692793,0.889829,0.935115,0.735181,0.601692,0.585830,0.874495,0.891450,0.724630,0.818651,0.933612,0.929505,0.860040,0.859452,0.917123,0.716937,0.544539,0.902214,0.897274,0.759024,0.888558,0.589987,0.934561,0.910678,0.762653,0.635642,0.730004,0.851004,0.596533,0.905578,0.885412
Arizona,0.922879,0.933834,NaN,0.928383,0.892733,0.834536,0.925722,0.597160,0.945849,0.927109,0.918501,0.915448,0.610604,0.931804,0.934323,0.943797,0.940432,0.936112,0.925560,0.917723,0.912929,0.925992,0.594232,0.922798,0.953821,0.936891,0.873405,0.519624,0.733738,0.951831,0.945251,0.943383,0.619970,0.895777,0.835505,0.940724,0.915860,0.909786,0.928382,0.934315,0.936855,0.935679,0.746451,0.948466,0.849900,0.933046,0.877886,0.932748,0.941347,0.904211,0.930512,0.851330,0.696166,0.615953,0.911441,0.916747,0.678123,0.893897,0.847945,0.933275,0.903548,0.789422,0.578321,0.932375
Auburn,0.933028,0.940752,0.929295,NaN,0.496611,0.578321,0.930984,0.816764,0.690092,0.916244,0.937292,0.924730,0.578321,0.927707,0.769612,0.922996,0.360884,0.928005,0.892822,0.935089,0.932682,0.864667,0.920529,0.915464,0.636116,0.301716,0.933690,0.910094,0.807272,0.925036,0.908072,0.931171,0.905217,0.875886,0.855829,0.939503,0.915451,0.933407,0.935852,0.923673,0.597160,0.864556,0.926405,0.806695,0.910931,0.934462,0.947476,0.932238,0.936554,0.947536,0.935191,0.793084,0.902097,0.934623,0.908914,0.795227,0.907090,0.942743,0.868408,0.919991,0.932493,0.943243,0.826833,0.921485
Baylor,0.878751,0.898735,0.909675,0.933393,NaN,0.702321,0.946633,0.924934,0.845732,0.904171,0.923188,0.686668,0.798367,0.836096,0.894411,0.771460,0.889688,0.939786,0.638798,0.930735,0.943411,0.898221,0.737829,0.924832,0.928374,0.931961,0.905987,0.940718,0.948973,0.732032,0.726202,0.900689,0.923243,0.861375,0.908850,0.782912,0.838844,0.915598,0.909759,0.917102,0.917629,0.630623,0.918458,0.928519,0.676640,0.714193,0.642416,0.814790,0.840721,0.618887,0.936332,0.910338,0.628577,0.907512,0.754244,0.912345,0.533875,0.855797,0.896282,0.946125,0.913826,0.861517,0.931307,0.704228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WKU,0.716640,0.294086,0.414739,0.307633,0.047961,0.106475,0.148439,0.641298,0.509095,0.101202,0.334425,0.402049,0.355461,0.472572,0.374268,0.185214,0.660055,0.455108,0.

In [61]:
df_matrix.to_csv(f'../data/simulations/mens/matchup_matrix_{season}.csv', index=True)
df_matrix_display.to_csv(f'../data/simulations/mens/matchup_matrix_display_{season}.csv', index=True)

'Done'

'Done'